# Classification Trees, Random Forest and Gradient Boosting – Titanic

**Dataset:** 891 passengers of the Titanic, which sank on 15 April 1912 after hitting an iceberg (Kaggle).
File: `../00_Data/titanic.csv`. This is the data set behind the classification examples of the concept notebooks.

| variable | meaning |
|---|---|
| `Survived` | 1 = survived, 0 = died – **target** |
| `Pclass` | ticket class (1 = first, 2 = second, 3 = third) |
| `Sex`, `Age` | sex, age in years (missing for 177 passengers) |
| `SibSp`, `Parch` | number of siblings/spouses and parents/children aboard |
| `Fare` | ticket price (British pounds) |
| `Embarked` | port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |
| `Name`, `Ticket`, `Cabin` | not used here |

**Question:** Which passengers had the best chances of survival? How well can survival be predicted from sex, age,
class and fare?

**After this exercise you can ...**
- compute Gini impurity and the impurity decrease of a split by hand,
- fit, visualise, interpret and prune a classification tree and read class probabilities from its leaves,
- explain how a random forest averages class probabilities,
- tune random forest and gradient boosting classifiers with `GridSearchCV` and compare them with ROC/AUC.

## Concept

The concepts behind this exercise (formulas, worked examples and additional explanations) are
explained in the notebook [classification_trees_ensembles_concept.ipynb](classification_trees_ensembles_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.inspection import permutation_importance

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Gini impurity.** A node contains 200 passengers, 76 of whom survived. Two candidate splits are evaluated:

| candidate split | left node (yes) | right node (no) |
|---|---|---|
| `male?` | 130 passengers: 25 survived / 105 not | 70 passengers: 51 survived / 19 not |
| `1st class?` | 50 passengers: 30 survived / 20 not | 150 passengers: 46 survived / 104 not |

Compute the Gini impurity of the parent node, $G_L$, $G_R$, the weighted impurity and $\Delta G$ for both candidate
splits. Which split does the tree choose?

**b) Leaf probability.** A leaf contains 145 survivors and 125 non-survivors (example from the concept notebook). What does it predict for a
new passenger who ends up in this leaf (class and probability)?

**c) Random forest.** Five trees give the probabilities of survival 0.9, 0.7, 0.4, 0.6 and 0.3 for a passenger. What
does the forest predict? What would a classic majority vote predict?

**d)** Verify your results with Python.

In [ ]:
# Exercise 1: Gini impurity by hand


# Function (to complete): computes the Gini impurity from the class counts
def gini(n_survived, n_died):
    """Gini impurity of a node with two classes."""
    # Your code here
    pass


# a) Gini impurity of the parent node and of both splits
# Your code here

# b) Leaf probability
# Your code here

# c) Random forest: mean probability vs. majority vote
# Your code here

**Your answer:** ...

## Exercise 2: Import and explore the data

a) Import the data. How many passengers survived? Which variables have missing values?

b) Compute the survival rate by `Sex`, by `Pclass` and by `Sex` × `Pclass`. Plot the survival rate by age group
(0–9, 10–19, …, 70–79 years) for women and men.

c) Prepare the features: `male` (1 = male), `Age` (missing values replaced by the median age), `Pclass`, `SibSp`,
`Parch`, `Fare` and dummy variables for `Embarked` (missing values replaced by `'S'`). Split the data into a stratified
training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data as titanic: '../00_Data/titanic.csv'
#    How many passengers survived? Which variables have missing values?
# Your code here

# b) Survival rates
# Your code here

# c) Features X, target y and stratified train/test split
# Your code here
X_train, X_test, y_train, y_test = ___

**Your answer:** ...

## Exercise 3: The example tree

Section «Anatomy of a classification tree» of the concept notebook shows a tree with the questions *male?*, *age ≤ 9.5?*, *3rd class?* and
*fare ≤ 23?*.

a) Fit a `DecisionTreeClassifier(max_depth=3, random_state=42)` on **all 891 passengers** with the features `male`,
`Age`, `third_class` (1 = 3rd class) and `Fare`, and plot it with `tree.plot_tree(..., class_names=['died', 'survived'])`.

b) Compare the tree with the example tree: which questions and thresholds are the same? Determine depth (`get_depth()`), size
(`tree_.node_count`) and number of leaves.

c) Check the root split by hand: compute the Gini impurity of the root and of its two children from the class counts
printed below, and $\Delta G$.

d) Use the tree to predict the survival probability of (1) a 30-year-old woman in 3rd class with a fare of 10 pounds and
(2) a 5-year-old boy in 2nd class with a fare of 30 pounds. Which leaves do they end up in?

In [ ]:
# a) Fit and plot the tree
X_example = pd.DataFrame(
    {
        'male': X['male'],
        'Age': X['Age'],
        'third_class': (X['Pclass'] == 3).astype(int),
        'Fare': X['Fare'],
    }
)
example_tree = ___
# Your code here

# b) Depth, size and number of leaves
# Your code here

# c) Class counts of the root and its children
t = example_tree.tree_
for node, name in [
    (0, 'root'),
    (t.children_left[0], 'left child'),
    (t.children_right[0], 'right child'),
]:
    counts = np.round(t.value[node][0] * t.n_node_samples[node]).astype(int)
    print(
        f'{name:12s}: n = {t.n_node_samples[node]}, died = {counts[0]}, '
        f'survived = {counts[1]}, Gini = {t.impurity[node]:.3f}'
    )

# d) Predictions for two passengers
new_passengers = pd.DataFrame(
    {'male': [0, 1], 'Age': [30, 5], 'third_class': [1, 0], 'Fare': [10, 30]}
)
# Your code here

**Your answer:** ...

## Exercise 4: Overfitting and pruning

From now on we use all prepared features and the training/test split of Exercise 2.

a) Fit a tree with `max_depth=3` and a tree without depth limit on the training set. Compare training accuracy, test
accuracy and test AUC. How many leaves does the full tree have?

b) Choose `ccp_alpha` with `GridSearchCV` (stratified 5-fold CV, `scoring='roc_auc'`) from the values of
`cost_complexity_pruning_path(X_train, y_train).ccp_alphas`. How many leaves does the best tree have, and what is its test
AUC?

In [ ]:
# Exercise 4: overfitting and pruning
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) Depth-3 tree vs. full tree
# Your code here

# b) Pruning with GridSearchCV
# Your code here

**Your answer:** ...

## Exercise 5: Look inside a random forest

a) Fit a `RandomForestClassifier(n_estimators=500, min_samples_leaf=5, oob_score=True, random_state=42)` on the
training set (`min_samples_leaf=5`: each leaf contains at least 5 passengers, so every tree outputs class shares) and print the
OOB accuracy (`oob_score_`) and the test accuracy.

b) Take the first passenger of the test set. Print the survival probabilities of the first 10 trees
(`rf.estimators_[i].predict_proba(...)`) and the mean over **all** 500 trees. Compare the mean with
`rf.predict_proba()`.

In [ ]:
# a) Random forest with OOB score
rf = ___
# Your code here

# b) Probabilities of single trees for one passenger
passenger = X_test.iloc[[0]]
# Your code here

**Your answer:** ...

## Exercise 6: «Putting it together in scikit-learn»

a) Run the code of section «Putting it together in scikit-learn» of the concept notebook on the Titanic data: a `GridSearchCV` for a
`RandomForestClassifier(random_state=42)` with `{'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 5, 20]}`,
`cv=5`, `scoring='roc_auc'`; then the test AUC and the confusion matrix for the threshold 0.5.

b) Do the same for a `GradientBoostingClassifier(random_state=42)` with
`{'learning_rate': [0.05, 0.1], 'n_estimators': [100, 300], 'max_depth': [2, 3]}`.

c) Which hyperparameters were selected? Interpret.

In [ ]:
# a) Random forest as in the concept notebook
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 5, 20]},
    cv=5,
    scoring='roc_auc',
)
# Your code here

# b) Gradient boosting
grid_gb = ___
# Your code here

**Your answer:** ...

## Exercise 7: Model comparison and interpretation

a) Plot the ROC curves of the depth-3 tree, the pruned tree, the tuned random forest and the tuned gradient boosting on
the test set and print the AUCs.

b) Compute the **permutation importance** of the tuned random forest on the test set (`scoring='roc_auc'`, 20 repeats)
and plot it.

c) What do the results tell us about the evacuation of the Titanic?

In [ ]:
# Exercise 7: compare the models on the test set
models = {
    'Tree (depth 3)': tree_3,
    'Pruned tree': pruned_tree,
    'Random forest': grid.best_estimator_,
    'Gradient boosting': grid_gb.best_estimator_,
}

# a) ROC curves and AUC
# Your code here

# b) Permutation importance
# Your code here

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')